# BSL Fingerspelling: training and honest evaluation

This notebook trains the model behind the **Fingerspelling** mode of our BSL Translator website.

**Input:** MediaPipe hand landmarks for two hands (2 hands × 21 points × x, y, z = 126 numbers).
**Output:** which BSL fingerspelling letter is being signed.

Steps:
1. Load and look at the data
2. Turn landmarks into features that don't depend on where you stand or how big your hands look on camera
3. Compare a **baseline** with three models on **held-out test data**
4. See **where the model fails** (confusion matrix)
5. Test on **our own hands**: people the model has never seen
6. Export the model for the website

**Dataset:** [BSL Numbers & Alphabet Hand Position For MediaPipe](https://www.kaggle.com/datasets/erentatepe/bsl-numbers-and-alphabet-hand-position-for-mediapipe) (Kaggle), file `two_hand_dataset.csv`.

## 1. Load the data

**On Kaggle:** click *Add Input* → search for the dataset above → add it. The code finds the file automatically.
**On Colab:** run the cell; if the file isn't found, it asks you to upload `two_hand_dataset.csv` (or the .zip).

In [ ]:
import glob, os, zipfile, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_csv(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True) + glob.glob(f'**/{name}', recursive=True)
    return hits[0] if hits else None

CSV = find_csv('two_hand_dataset.csv')
if CSV is None:
    try:
        from google.colab import files
        print('Upload two_hand_dataset.csv (or the .zip)')
        for fname in files.upload():
            if fname.endswith('.zip'):
                zipfile.ZipFile(fname).extractall('.')
        CSV = find_csv('two_hand_dataset.csv')
    except ImportError:
        pass
assert CSV, 'two_hand_dataset.csv not found'
print('Using', CSV)

df = pd.read_csv(CSV, header=None)          # the file has no header row
raw = df.iloc[:, :126].values               # 126 landmark numbers per row
y = df[126].str.split(' - ').str[1].str.upper().values   # 'A - a' -> 'A', 'Ten - 10' -> '10'
print(df.shape[0], 'rows,', len(np.unique(y)), 'classes:', ' '.join(np.unique(y)))

In [ ]:
counts = pd.Series(y).value_counts().sort_index()
counts.plot.bar(figsize=(10, 3), color='#0b6e69', title='Examples per letter')
plt.ylabel('rows'); plt.show()
print('Smallest class:', counts.idxmin(), counts.min(), '| largest:', counts.idxmax(), counts.max())

**What's missing:** C, H, J and Y are not in this file. C is signed with one hand in BSL, and the dataset
author left out H, J and Y because they involve movement. Our website can't recognise those letters,
and we say so in the write-up.

Let's look at a few examples to check the data makes sense (always look before training).

In [ ]:
HAND = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),(5,9),(9,10),(10,11),(11,12),
        (9,13),(13,14),(14,15),(15,16),(13,17),(17,18),(18,19),(19,20),(0,17)]
def plot_hands(ax, r, title):
    pts = r.reshape(2, 21, 3)
    for h, col in zip(pts, ['#f59e0b', '#14b8a6']):
        for a, b in HAND:
            ax.plot([h[a,0], h[b,0]], [h[a,1], h[b,1]], color=col, lw=2)
    ax.set_title(title); ax.invert_yaxis(); ax.set_aspect('equal'); ax.axis('off')

show = ['A', 'B', 'D', 'L', 'M', 'V']
fig, axes = plt.subplots(1, len(show), figsize=(15, 3))
for ax, L in zip(axes, show):
    plot_hands(ax, raw[np.where(y == L)[0][0]], L)
plt.show()

## 2. Features

Raw coordinates depend on where the person stands and how close they are to the camera. We fix that:

1. **Order the two hands** by which wrist is further left, so "hand 1" always means the same thing
2. **Centre** on the average of all 42 points
3. **Scale** by the typical distance of the points from that centre

The website does exactly the same steps in JavaScript, so the model sees the same kind of numbers live.

**Augmentation** (training data only): we add mirrored copies (so left-handed signers and mirrored cameras
still work), slightly stretched copies (different camera shapes) and a little noise.

In [ ]:
def features(raw):
    R = raw.reshape(-1, 2, 21, 3).astype(np.float64).copy()
    swap = R[:, 0, 0, 0] > R[:, 1, 0, 0]                 # hand with the leftmost wrist first
    R[swap] = R[swap][:, ::-1]
    c = R[:, :, :, :2].reshape(-1, 42, 2).mean(1)        # centre of all points
    P = R.copy()
    P[:, :, :, 0] -= c[:, None, None, 0]
    P[:, :, :, 1] -= c[:, None, None, 1]
    s = np.sqrt((P[:, :, :, :2] ** 2).sum(-1).mean((1, 2)))   # size
    s[s == 0] = 1
    P /= s[:, None, None, None]
    return P.reshape(len(P), -1)

def augment(raw, rng, flip=True):
    R = raw.reshape(-1, 2, 21, 3).copy()
    if flip:
        R[:, :, :, 0] = 1 - R[:, :, :, 0]                  # mirror image
    sx = rng.uniform(0.8, 1.25, size=(len(R), 1, 1))      # stretch sideways
    R[:, :, :, 0] = 0.5 + (R[:, :, :, 0] - 0.5) * sx
    R[:, :, :, :2] += rng.normal(0, 0.003, size=R[:, :, :, :2].shape)   # small noise
    return R.reshape(len(R), -1)

def with_augmentation(raw_part, y_part, seed=0):
    rng = np.random.default_rng(seed)
    X = np.vstack([features(raw_part), features(augment(raw_part, rng, True)), features(augment(raw_part, rng, False))])
    return X, np.concatenate([y_part] * 3)

## 3. Train/test split, baseline and models

**Why not a normal random split?** The rows look like they come from video frames: neighbouring rows of the
same letter are almost identical. A random split puts near-copies in both training and test, which makes
results look better than they really are.

So we use a **block split**: for each letter, the first 80% of its rows are training and the **last 20% are
test**. We never use the test set to tune anything.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

is_test = np.zeros(len(y), bool)
for L in np.unique(y):
    idx = np.where(y == L)[0]
    is_test[idx[int(len(idx) * 0.8):]] = True
tr, te = np.where(~is_test)[0], np.where(is_test)[0]
Xtr, Xte = features(raw[tr]), features(raw[te])
print(len(tr), 'training rows,', len(te), 'test rows')

models = {
    'Baseline (always guess the most common letter)': (DummyClassifier(strategy='most_frequent'), False),
    'Logistic regression': (LogisticRegression(max_iter=2000), False),
    'Random forest (300 trees)': (RandomForestClassifier(300, n_jobs=-1, random_state=0), False),
    'Neural network (MLP 128-64) + augmentation': (MLPClassifier((128, 64), max_iter=300, early_stopping=True, random_state=0), True),
}
results, fitted = {}, {}
for name, (m, aug) in models.items():
    X_, y_ = with_augmentation(raw[tr], y[tr]) if aug else (Xtr, y[tr])
    m.fit(X_, y_)
    fitted[name] = m
    results[name] = accuracy_score(y[te], m.predict(Xte))
pd.DataFrame({'test accuracy': results}).style.format('{:.1%}')

## 4. Where does it go wrong?

Accuracy on this test set is very high. That's a warning sign, not a victory: the dataset probably comes
from **very few signers**, so the test rows are still the same people's hands. The confusion matrix shows
which letters get mixed up; the real test is section 5.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
best = 'Neural network (MLP 128-64) + augmentation'
pred = fitted[best].predict(Xte)
fig, ax = plt.subplots(figsize=(9, 9))
ConfusionMatrixDisplay.from_predictions(y[te], pred, ax=ax, colorbar=False, cmap='Greens')
ax.set_title(best + ' on held-out test rows'); plt.show()

labels = np.unique(y)
cm = confusion_matrix(y[te], pred, labels=labels)
np.fill_diagonal(cm, 0)
pairs = sorted(((cm[i, j], labels[i], labels[j]) for i in range(len(labels)) for j in range(len(labels)) if cm[i, j]), reverse=True)[:8]
print('Most common mistakes (true -> predicted):')
for n, a, b in pairs:
    print(f'  {a} -> {b}: {n}')

## 5. The honest test: our own hands

On the website, switch to **Fingerspelling** → **Train** tab → record yourself signing letters →
**Export as CSV**. Do this for each teammate (ideally different hand sizes, skin tones, lighting, and a
left-handed signer if possible) and name the files e.g. `test_alex.csv`, `test_sam.csv`.

Upload them here. These are people the model has **never** seen, so this is the number to report.

In [ ]:
own_files = glob.glob('/kaggle/input/**/test_*.csv', recursive=True) + glob.glob('test_*.csv') + glob.glob('my_test_letters*.csv')
if not own_files:
    try:
        from google.colab import files
        print('Upload your exported test CSVs (or press Cancel to skip)')
        own_files = list(files.upload().keys())
    except Exception:
        pass

final_model = fitted[best]
rows = []
for f in own_files:
    d = pd.read_csv(f, header=None)
    yy = d[126].str.split(' - ').str[1].str.upper().values
    pp = final_model.predict(features(d.iloc[:, :126].values))
    rows.append({'file': os.path.basename(f), 'frames': len(d), 'accuracy': (pp == yy).mean()})
    per_letter = pd.Series(pp == yy, index=yy).groupby(level=0).mean().sort_values()
    print(f'\n{os.path.basename(f)}: weakest letters\n', per_letter.head(5).map('{:.0%}'.format).to_string())
if rows:
    display(pd.DataFrame(rows).style.format({'accuracy': '{:.1%}'}))
else:
    print('No own test files yet. Record some on the website, then run this cell again.')

## 6. Export the model for the website

We retrain the neural network on **all** the data (with augmentation) and save its weights as
`fingerspell_model.json`. Upload that file to the GitHub repo next to `index.html`.

In [ ]:
X_all, y_all = with_augmentation(raw, y)
final = MLPClassifier((128, 64), max_iter=300, early_stopping=True, random_state=0).fit(X_all, y_all)
r5 = lambda a: np.round(a, 5).tolist()
model = {
    'format': 'bsl-fingerspell-mlp-v1',
    'features': 'two hands sorted by wrist x; centred on mean of 42 points; scaled by RMS distance (x,y,z)',
    'assumed_aspect': 4 / 3,
    'classes': final.classes_.tolist(),
    'layers': [{'W': r5(W), 'b': r5(b)} for W, b in zip(final.coefs_, final.intercepts_)],
    'trained_on': 'Kaggle: BSL Numbers & Alphabet Hand Position For MediaPipe (two_hand_dataset.csv)',
}
with open('fingerspell_model.json', 'w') as f:
    json.dump(model, f, separators=(',', ':'))
print('Saved fingerspell_model.json,', os.path.getsize('fingerspell_model.json') // 1024, 'KB')
try:
    from google.colab import files; files.download('fingerspell_model.json')
except Exception:
    print('On Kaggle: find it in the Output panel on the right.')

## For the write-up

- **Baseline vs model:** copy the table from section 3.
- **The honest number:** accuracy on your own recordings (section 5). It will be lower than section 3; explain why
  (few signers in the dataset, different cameras and lighting).
- **Where it fails:** the most common mistakes from section 4 and the weakest letters from section 5.
- **Limitations:** no C, H, J, Y; static handshapes only; trained on few signers. Note whether it works equally
  well for every teammate.